In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import string
import pickle

In [2]:
with open('../static/model/model.pickle', 'rb') as f:
    model = pickle.load(f)

In [3]:
import nltk
with open('../static/model/nltk_data/corpora/stopwords/english', 'r') as file:
    sw = file.read().splitlines()

In [4]:
vocab = pd.read_csv('../static/model/vocablary.txt', header=None)
tokens = vocab[0].tolist()

In [5]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [6]:
def preprocessing(text):
    data = pd.DataFrame([text], columns=['statement'])
    data["statement"] = data["statement"].astype(str).apply(lambda x: " ".join(word.lower() for word in x.split()))
    
    data["statement"] = data["statement"].astype(str).apply(
        lambda x: " ".join(re.sub(r'https?://\S+|www\.\S+', '', word)for word in x.split()))
    
    data["statement"] = data["statement"].astype(str).apply(
        lambda x: x.translate(str.maketrans('', '', string.punctuation)))
    
    data["statement"] = data["statement"].astype(str).apply(
        lambda x: re.sub(r'\d+', '', x))
    
    data["statement"] = data["statement"].astype(str).apply(
        lambda x: " ".join(word for word in x.split() if word.lower() not in sw))
    
    data["statement"] = data["statement"].astype(str).apply(
        lambda x: " ".join(ps.stem(x) for x in x.split()))

    return data ["statement"]

In [7]:
def vectorizer(ds, vocabulary):
    vectorized_lst = []

    for sentence in ds:
        sentence_lst = np.zeros(len(vocabulary))

        for i in range(len(vocabulary)):
            if vocabulary[i] in sentence.split():
                sentence_lst[i] = 1

        vectorized_lst.append(sentence_lst)

    vectorized_lst_new = np.asarray(vectorized_lst, dtype = np.float32)

    return vectorized_lst_new

In [8]:
def get_prediction(vectorized_txt):
    prediction = model.predict(vectorized_txt)[0]   

    if prediction == "Anxiety":
        return "Anxiety"
    elif prediction == "Normal":
        return "Normal"
    elif prediction == "Suicidal":
        return "Suicidal"
    elif prediction == "Depression":
        return "Depression"
    elif prediction == "Stress":
        return "Stress"
    elif prediction == "Bipolar":
        return "Bipolar"
    elif prediction == "Personality disorder":
        return "Personality disorder"
    else:
        return "None"

In [9]:
txt = "I want to die"
preprocessed_txt = preprocessing(txt)
vectorized_txt = vectorizer(preprocessed_txt, tokens)
predict = get_prediction(vectorized_txt)

In [10]:
predict


'Suicidal'